In [57]:
import os
import sys
import requests
import openai
import csv
import time
from pathlib import Path
from dotenv import load_dotenv
# Add parent directory to sys.path
parent_dir = os.path.abspath("..")
sys.path.append(parent_dir)
from backend.constants import (
    ANKI_CONNECT_URL
)
env_path = Path.cwd().parent / "backend" / ".env"
print(env_path)
load_dotenv(dotenv_path=env_path)

/Users/sethdonaldson/sourcecode/anki-vocab-generator/backend/.env


True

In [58]:


# ---------------------- Configuration ----------------------
DECK_NAME = "8000+ most common swedish words"
MODEL_NAME = "Memrise - 8000+ Most Common Swedish Words - Part 1 (of four) - Swedish"
VOCAB_FIELD = "Swedish"
DEFINITION_FIELD = "English"
EX_SENTENCE_SW_FIELD = "Example Sentence (Swedish)"
EX_SENTENCE_EN_FIELD = "Example Sentence (English)"

# CSV output file
CSV_OUTPUT_FILE = "swedish_examples.csv"

# ---------------------- AnkiConnect Helper Functions ----------------------
def request(action, **params):
    return {"action": action, "params": params, "version": 6}

def invoke(action, **params):
    payload = request(action, **params)
    response = requests.post(ANKI_CONNECT_URL, json=payload)
    if not response.ok:
        raise Exception(f"Request failed with status code {response.status_code}: {response.text}")
    response_json = response.json()
    if 'error' not in response_json or 'result' not in response_json:
        raise Exception('Invalid response structure')
    if response_json['error'] is not None:
        raise Exception(response_json['error'])
    return response_json['result']

# ---------------------- LLM Generation Function ----------------------
def generate_examples(vocab_word, definition):
    """
    Using the vocabulary word (in Swedish) as context,
    generate a Swedish example sentence and its English translation.
    
    Returns a tuple: (swedish_sentence, english_sentence)
    """
    prompt = (f"""Generate a Swedish example sentence that naturally uses the word '{vocab_word}' with the given definition: '{definition}'.
              
              Keep it simple, simple, and memorable. The purpose is to help the user remember the word, as the sentence 
              (and its english translation) will be used in an Anki flashcard for learning the vocabulary word.
              I am a beginner trying to learn Swedish, so make sure the sentence is simple and easy to understand.
              Don't be afraid to use common Swedish phrases and idioms (as appropriate). The purpose is to help the user learn common, spoken/written Swedish.
              
              THEN provide the English translation for that sentence on a new line.
              Return the answer in exactly two lines: first the Swedish sentence, then the English sentence. With no other text.
              
              IMPORTANT: if the vocabulary word is a verb, it will appear as "att" + the verb. This doesn't necessarily mean you should use "att" in your example sentence.
              Just use the verb as you would in a normal sentence.
              IMPORTANT: if the vocabulary word is a noun, it will appear as "en" or "ett" + the noun. This doesn't necessarily mean you should use "en" or "ett" in your example sentence.
              Just use the noun as you would in a normal sentence.
              
              EXAMPLE: If the vocab word = vatten
              You would output something like this:
              Vatten är en vätska
              Water is a liquid
              """)
    try:
        client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY")) 
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You generate natural-sounding Swedish sentences and their English translations."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=60,
        )
        
        # Split output into two lines (we expect exactly 2 lines).
        output = response.choices[0].message.content.strip().split("\n")
        if len(output) < 2:
            raise Exception("Unexpected output format from LLM")
        swedish_sentence = output[0].strip()
        english_sentence = output[1].strip()
        return swedish_sentence, english_sentence
    except Exception as e:
        print(f"Error generating example for '{vocab_word}': {e}")
        return None, None


In [59]:
print(OPENAI_API_KEY)

None


In [60]:
# ---------------------- Main Process ----------------------
def main():
    # Step 1: Get all note IDs for the specified deck.
    query = f'deck:"{DECK_NAME}"'
    note_ids = invoke("findNotes", query=query)
    if not note_ids:
        print(f"No notes found in deck {DECK_NAME}")
        return
    
    # Get note info including fields.
    notes_info = invoke("notesInfo", notes=note_ids)
    
    # Prepare a list to optionally store results in CSV.
    csv_rows = [["Vocabulary Word", "Example Sentence (Swedish)", "Example Sentence (English)"]]
    
    total_notes = len(notes_info)
    print(f"Processing {total_notes} notes...")
    
    for i, note in enumerate(notes_info, start=1):
        note_id = note["noteId"] if "noteId" in note else note["id"]
        fields = note["fields"]
        vocab_word = fields.get(VOCAB_FIELD, {}).get("value", "").strip()
        definition = fields.get(DEFINITION_FIELD, {}).get("value", "").strip()
        if not vocab_word:
            print(f"Note ID {note_id} does not have a value for '{VOCAB_FIELD}'. Skipping.")
            continue
        
        # swedish_sentence = fields.get(EX_SENTENCE_SW_FIELD, {}).get("value", "").strip()
        # if swedish_sentence:
        #     print(f"[{i}/{total_notes}] Skipping note ID {note_id} because it already has an example sentence.")
        #     continue
        
        
        print(f"[{i}/{total_notes}] Generating examples for '{vocab_word}'")
        swedish_sentence, english_sentence = generate_examples(vocab_word, definition)
        
        if not swedish_sentence or not english_sentence:
            print(f"Skipping note ID {note_id} due to generation error.")
            continue
        
        # Update CSV row list.
        csv_rows.append([vocab_word, swedish_sentence, english_sentence])
        
        # Update the note's fields with the generated content.
        update_payload = {
            "id": note_id,
            "fields": {
                EX_SENTENCE_SW_FIELD: swedish_sentence,
                EX_SENTENCE_EN_FIELD: english_sentence
            }
        }
        print(update_payload)
        
        try:
            invoke("updateNoteFields", note=update_payload)
        except Exception as e:
            print(f"Error updating note ID {note_id}: {e}")
        
        # Optional: add a short delay to avoid overloading the API (adjust as needed).
        # time.sleep(1)
    
    # Write CSV file (optional)
    with open(CSV_OUTPUT_FILE, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(csv_rows)
    print(f"CSV file saved to {CSV_OUTPUT_FILE}")

In [61]:
main()

Processing 8327 notes...
[1/8327] Skipping note ID 1598022787258 because it already has an example sentence.
[2/8327] Skipping note ID 1598022787263 because it already has an example sentence.
[3/8327] Skipping note ID 1598022787266 because it already has an example sentence.
[4/8327] Skipping note ID 1598022787269 because it already has an example sentence.
[5/8327] Skipping note ID 1598022787273 because it already has an example sentence.
[6/8327] Skipping note ID 1598022787277 because it already has an example sentence.
[7/8327] Skipping note ID 1598022787280 because it already has an example sentence.
[8/8327] Skipping note ID 1598022787283 because it already has an example sentence.
[9/8327] Skipping note ID 1598022787286 because it already has an example sentence.
[10/8327] Skipping note ID 1598022787290 because it already has an example sentence.
[11/8327] Skipping note ID 1598022787293 because it already has an example sentence.
[12/8327] Skipping note ID 1598022787295 because 

KeyboardInterrupt: 